In [ ]:
pip install datasets optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.6/386.6 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.5/242.5 kB 21.0 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
path = "/content/drive/MyDrive/Colab Notebooks/data/"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
df = pd.read_csv(path+"output.csv")
print(len(df))
df = df[df['price_change_4h'].notna()]
df = df[df['volume_surge_price'].notna()]
print(len(df))
df

20258
20251


,timestamp,title,source,volume_surge_price,price_change_4h
0,1629741303,Visa Makes Their Grand Entrance Into the NFT A...,Watcher Guru,-1.006608e-06,-0.000297
1,1629936472,Have you seen our billboard in Times Square to...,Watcher Guru,-2.814638e-03,-0.013606
2,1630005294,Record breaking sale for Bored Ape,Watcher Guru,6.691295e-03,0.013199
3,1630605534,"This is our only Telegram, please be careful o...",Watcher Guru,-9.948841e-07,0.000483
4,1631232301,Youtube Sensation MrBeast Reveals His Portfoli...,Watcher Guru,2.904352e-04,0.000779
...,...,...,...,...,...
20246,1743146885,NY AG files $200 million settlement with Galax...,unfolded.,-1.440233e-02,-0.009280
20247,1743169126,Terraform Labs to open crypto creditor claims ...,unfolded.,-7.996360e-02,-0.010471
20248,1743181914,FDIC clarifies that banks can engage in permit...,unfolded.,-2.301976e-02,-0.006169
20249,1743188025,Trump pardons three co-founders of BitMex cryp...,unfolded.,-2.509775e-04,0.000081


Init and modify FinBERT for regression

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer

class FinBERTRegression(nn.Module):
    def __init__(self, model_name="yiyanghkust/finbert-tone"):
        super(FinBERTRegression, self).__init__()
        self.bert = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)
        self.regressor = nn.Linear(3, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        regression_score = self.regressor(outputs.logits)
        normalized_score = torch.tanh(regression_score)
        return normalized_score

class AdaptiveWeightedHuberLoss(nn.Module):
    def __init__(self, delta=1.0):
        super(AdaptiveWeightedHuberLoss, self).__init__()
        self.delta = delta

    def forward(self, predictions, targets, weights=None):
        error = predictions - targets
        abs_error = torch.abs(error)
        quadratic = torch.minimum(abs_error, torch.tensor(self.delta))
        linear = abs_error - quadratic
        loss = 0.5 * quadratic ** 2 + self.delta * linear
        if weights is not None:
            loss = loss * weights
        return loss.mean()



# Load model & tokenizer
model = FinBERTRegression("yiyanghkust/finbert-tone")
tokenizer = AutoTokenizer.from_pretrained("yiyanghkust/finbert-tone")

#Freeze all BERT layers
for param in model.bert.parameters():
    param.requires_grad = False

# Unfreeze only the last 3 layers of BERT
for layer in model.bert.bert.encoder.layer[-2:]:
    for param in layer.parameters():
        param.requires_grad = True

# Ensure the classifier head is not trainable
for param in model.regressor.parameters():
    param.requires_grad = False

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


FinBERTRegression(
  (bert): BertForSequenceClassification(
    (bert): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(30873, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (token_type_embeddings): Embedding(2, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (encoder): BertEncoder(
        (layer): ModuleList(
          (0-11): 12 x BertLayer(
            (attention): BertAttention(
              (self): BertSdpaSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): BertSelfOutput(
                (dense): Linear(in_features=768, out_feature

Data split

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from datetime import timedelta


class SentimentDataset(Dataset):
    def __init__(self, texts, scores, tokenizer, max_length=128):
        self.texts = texts
        self.scores = scores
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        score = self.scores[idx]

        encoded = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "labels": torch.tensor(score, dtype=torch.float).unsqueeze(0)  # Ensure correct shape
        }

# train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)
# print(len(train_df))

# train_set = SentimentDataset(train_df["title"].tolist(), train_df["volume_surge_price"].tolist(), tokenizer)
# train_loader = DataLoader(train_set, batch_size=8, shuffle=True)

# test_set = SentimentDataset(test_df["title"].tolist(), test_df["volume_surge_price"].tolist(), tokenizer)
# test_loader = DataLoader(test_set, batch_size=8, shuffle=False)

df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')

# Sort by timestamp just in case
df = df.sort_values('timestamp')

# Define the cutoff date for 365 days before the latest timestamp
cutoff_date = df['timestamp'].max() - timedelta(days=365)
print(df['timestamp'].max())
print(cutoff_date)

# Split the data
train_df = df[df['timestamp'] < cutoff_date]
test_df = df[df['timestamp'] >= cutoff_date]

# Print sizes for sanity check
print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")

# Create datasets and dataloaders
train_set = SentimentDataset(train_df["title"].tolist(), train_df["volume_surge_price"].tolist(), tokenizer)
train_loader = DataLoader(train_set, batch_size=8, shuffle=True)

test_set = SentimentDataset(test_df["title"].tolist(), test_df["volume_surge_price"].tolist(), tokenizer)
test_loader = DataLoader(test_set, batch_size=8, shuffle=False)

2025-03-28 21:11:59
2024-03-28 21:11:59
Train size: 17570, Test size: 2681


In [ ]:
import torch.nn.functional as F
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr
import numpy as np



def evaluate(model, test_loader, device):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask)
            predictions = outputs.squeeze().cpu().numpy()

            loss = F.mse_loss(outputs.squeeze(), labels.squeeze())
            total_loss += loss.item()
            #predictions = [predictions]
            predictions = np.array(predictions)

            if predictions.ndim == 0:
                predictions = [predictions.item()]  # Convert scalar to list
            else:
                predictions = predictions.tolist()

            all_preds.extend(predictions)
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(test_loader)
    print(f"Test MSE Loss: {avg_loss:.4f}")

    return all_preds, all_labels

In [ ]:


def compute_metrics(predictions, actuals, df, metric_label="Model"):
    predictions = np.array(predictions)
    actuals = np.array(actuals)

    if len(predictions) == 0 or len(actuals) == 0:
        print(f"{metric_label} - Skipping metrics computation: empty predictions or actuals.")
        return {
            "Metric": metric_label,
            "MSE": None,
            "RMSE": None,
            "MAE": None,
            "MAPE": None,
            "SMAPE": None,
            "R²": None,
            "Corr": None,
            "Corr_P": None,
        }

    mse = mean_squared_error(actuals, predictions)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(actuals, predictions)

    epsilon = 1e-8  # Avoid division by zero
    mape = np.mean(np.abs((actuals - predictions) / (actuals + epsilon))) * 100
    smape = 100 * np.mean(2.0 * np.abs(predictions - actuals) / (np.abs(predictions) + np.abs(actuals) + epsilon))
    r2 = r2_score(actuals, predictions)

    target_correlation = df.loc[df.index[:len(predictions)], 'price_change_4h'].values
    corr_with_price_change_4h, corr_p_val = pearsonr(predictions, target_correlation)

    print(f"{metric_label} Performance:")
    print(f"🔹 MSE:   {mse:.4f}")
    print(f"🔹 RMSE:  {rmse:.4f}")
    print(f"🔹 MAE:   {mae:.4f}")
    print(f"🔹 MAPE:  {mape:.2f}%")
    print(f"🔹 SMAPE: {smape:.2f}%")
    print(f"🔹 R²:    {r2:.4f}")
    print(f"🔹 Corr with price_change_4h: {corr_with_price_change_4h:.4f}")
    print(f"🔹 Corr p: {corr_p_val:.4f}")
    print("-" * 40)

    return {
        "Metric": metric_label,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "SMAPE": smape,
        "R²": r2,
        f"Corr": corr_with_price_change_4h,
        f"Corr_P": corr_p_val,
    }


In [ ]:
# import optuna
# from torch.utils.data import DataLoader
# from tqdm.notebook import tqdm


# def objective(trial):
#     # Search space
#     optimizer_name = "AdamW" # trial.suggest_categorical("optimizer", ["AdamW", "SGD", "RMSprop"])
#     lr = trial.suggest_float("lr", 1e-5, 0.001, log=True)
#     batch_size = 64 # trial.suggest_categorical("batch_size", [8, 16])
#     max_len = 128 # trial.suggest_categorical("max_len", [64, 128])
#     weight_decay = trial.suggest_float("weight_decay", 0.0005, 0.05)
#     momentum = 0#trial.suggest_float("momentum", 0.001, 0.4)

#     # Prepare datasets with trial-specific max_len
#     df_small = df.sample(n=3000, random_state=42)
#     train_df_small, test_df_small = train_test_split(df_small, test_size=0.1, random_state=42)
#     print(len(train_df))

#     train_set_small = SentimentDataset(train_df_small["title"].tolist(), train_df_small["volume_surge_price"].tolist(), tokenizer, max_length=max_len)
#     train_loader_small = DataLoader(train_set_small, batch_size=16, shuffle=True)

#     test_set_small = SentimentDataset(test_df_small["title"].tolist(), test_df_small["volume_surge_price"].tolist(), tokenizer, max_length=max_len)
#     test_loader_small = DataLoader(test_set_small, batch_size=16, shuffle=False)

#     # Reinitialize model
#     model = FinBERTRegression("yiyanghkust/finbert-tone")
#     for param in model.bert.parameters():
#         param.requires_grad = False

#     for layer in model.bert.bert.encoder.layer[-2:]:
#         for param in layer.parameters():
#             param.requires_grad = True

#     for param in model.regressor.parameters():
#         param.requires_grad = False

#     if optimizer_name == "AdamW":
#         optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
#     elif optimizer_name == "Adam":
#         optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
#     elif optimizer_name == "SGD":
#         optimizer = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay, momentum=momentum)
#     elif optimizer_name == "RMSprop":
#         optimizer = torch.optim.RMSprop(model.parameters(), lr=lr, weight_decay=weight_decay)

#     model.to(device)

#     loss_fn = AdaptiveWeightedHuberLoss()

#     # Train for 1 epoch (for speed)
#     model.train()
#     loop = tqdm(train_loader_small, leave=False)
#     for batch in loop:
#       input_ids = batch["input_ids"].to(device)
#       attention_mask = batch["attention_mask"].to(device)
#       labels = batch["labels"].to(device)

#       optimizer.zero_grad()
#       outputs = model(input_ids, attention_mask)
#       loss = loss_fn(outputs.squeeze(-1), labels.squeeze(-1))  # Ensures both are [8]

#       loss.backward()
#       optimizer.step()

#     # Evaluate
#     preds, labels = evaluate(model, test_loader_small, device)
#     metrics = compute_metrics(preds, labels, test_df, metric_label="trial")

#     return metrics["MSE"]  # Minimize MSE

# study = optuna.create_study(direction="minimize")
# study.optimize(objective, n_trials=100)

# print("\n🏆 Best Trial:")
# print(study.best_trial)


train

In [ ]:
from tqdm.notebook import tqdm

def train(model, train_loader, optimizer, loss_fn, device, epochs=5):
    model.train()

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        loop = tqdm(train_loader, leave=False)
        for batch in loop:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask)
            loss = loss_fn(outputs.squeeze(-1), labels.squeeze(-1))  # Ensures both are [8]

            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            loop.set_description(f"Epoch {epoch+1}")
            loop.set_postfix(loss=loss.item())

        print(f"Epoch {epoch+1}: Loss = {total_loss / len(train_loader):.4f}")

        predictions_finetuned, actuals = evaluate(model, test_loader, device)
        metrics_finetuned = compute_metrics(predictions_finetuned,actuals, test_df, "Fine-Tuned FinBERT")

# optimizer = torch.optim.AdamW([
#     {"params": model.bert.parameters(), "lr": 5e-5},  # Lower LR for pre-trained BERT
#     #{"params": model.regressor.parameters(), "lr": 0}  # Higher LR for regression layer
# ], weight_decay=0.01)

# optimizer = torch.optim.AdamW([
#     {"params": model.bert.parameters(), "lr": 0.000735},  # Lower LR for pre-trained BERT
#     #{"params": model.regressor.parameters(), "lr": 0}  # Higher LR for regression layer
# ], weight_decay=0.01, lr=0.00735)

optimizer = torch.optim.AdamW(model.parameters(), lr=0.0000735, weight_decay=0.094)


loss_fn = AdaptiveWeightedHuberLoss(delta=1)
train(model, train_loader, optimizer, loss_fn, device, epochs=100)


  0%|          | 0/2197 [00:00<?, ?it/s]

Epoch 1: Loss = 0.0075
Test MSE Loss: 0.0067
Fine-Tuned FinBERT Performance:
🔹 MSE:   0.0067
🔹 RMSE:  0.0821
🔹 MAE:   0.0469
🔹 MAPE:  36744.52%
🔹 SMAPE: 159.47%
🔹 R²:    -0.1112
🔹 Corr with price_change_4h: -0.0034
🔹 Corr p: 0.8602
----------------------------------------


  0%|          | 0/2197 [00:00<?, ?it/s]

Epoch 2: Loss = 0.0039
Test MSE Loss: 0.0062
Fine-Tuned FinBERT Performance:
🔹 MSE:   0.0063
🔹 RMSE:  0.0791
🔹 MAE:   0.0411
🔹 MAPE:  29085.80%
🔹 SMAPE: 157.11%
🔹 R²:    -0.0326
🔹 Corr with price_change_4h: 0.0753
🔹 Corr p: 0.0001
----------------------------------------


  0%|          | 0/2197 [00:00<?, ?it/s]

Epoch 3: Loss = 0.0029
Test MSE Loss: 0.0068
Fine-Tuned FinBERT Performance:
🔹 MSE:   0.0068
🔹 RMSE:  0.0823
🔹 MAE:   0.0370
🔹 MAPE:  17919.95%
🔹 SMAPE: 163.24%
🔹 R²:    -0.1186
🔹 Corr with price_change_4h: 0.0832
🔹 Corr p: 0.0000
----------------------------------------


... (34 outputs left out)

  0%|          | 0/2197 [00:00<?, ?it/s]

Epoch 21: Loss = 0.0011
Test MSE Loss: 0.0062
Fine-Tuned FinBERT Performance:
🔹 MSE:   0.0062
🔹 RMSE:  0.0785
🔹 MAE:   0.0328
🔹 MAPE:  13542.07%
🔹 SMAPE: 157.19%
🔹 R²:    -0.0175
🔹 Corr with price_change_4h: 0.0417
🔹 Corr p: 0.0307
----------------------------------------


  0%|          | 0/2197 [00:00<?, ?it/s]

KeyboardInterrupt: 

evaluate

In [ ]:
import pandas as pd
df = pd.read_csv(path+"output.csv")
print(len(df))
df = df[df['price_change_4h'].notna()]
df = df[df['volume_surge_price'].notna()]
print(len(df))
df

# train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)
# print(len(train_df))

# train_set = SentimentDataset(train_df["title"].tolist(), train_df["volume_surge_price"].tolist(), tokenizer)
# train_loader = DataLoader(train_set, batch_size=8, shuffle=True)

# test_set = SentimentDataset(test_df["title"].tolist(), test_df["volume_surge_price"].tolist(), tokenizer)
# test_loader = DataLoader(test_set, batch_size=8, shuffle=False)

# predictions_finetuned, actuals = evaluate(model, test_loader, device)
# metrics_finetuned = compute_metrics(predictions_finetuned,actuals, test_df, "Fine-Tuned FinBERT")


df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')

# Sort by timestamp just in case
df = df.sort_values('timestamp')

# Define the cutoff date for 365 days before the latest timestamp
cutoff_date = df['timestamp'].max() - timedelta(days=365)
print(df['timestamp'].max())
print(cutoff_date)

# Split the data
train_df = df[df['timestamp'] < cutoff_date]
test_df = df[df['timestamp'] >= cutoff_date]

# Print sizes for sanity check
print(f"Train size: {len(train_df)}, Test size: {len(test_df)}")

# Create datasets and dataloaders
train_set = SentimentDataset(train_df["title"].tolist(), train_df["volume_surge_price"].tolist(), tokenizer)
train_loader = DataLoader(train_set, batch_size=9, shuffle=True)

test_set = SentimentDataset(test_df["title"].tolist(), test_df["volume_surge_price"].tolist(), tokenizer)
test_loader = DataLoader(test_set, batch_size=9, shuffle=False)


# Run evaluation
predictions_finetuned, actuals = evaluate(model, test_loader, device)
#second = [np.array([x], dtype=np.float32) for arr in predictions_finetuned for x in arr]

print(predictions_finetuned)
print(actuals)

metrics_finetuned = compute_metrics(predictions_finetuned,actuals, test_df, "Fine-Tuned FinBERT")

In [ ]:
torch.save(model.state_dict(), 'model_weights_2.pth')

# model = FinBERTRegression()
# model.load_state_dict(torch.load(path+'model_weights_2.pth', map_location=torch.device('cpu'))) #
# model.eval()  # set to eval mode if you're not training

In [ ]:
# Load model & tokenizer
model2 = FinBERTRegression("yiyanghkust/finbert-tone")
model2.eval()
model2.to(device)
#predictions2, actuals2 = evaluate(model2, test_loader, device)
#metrics_base = compute_metrics(predictions2, actuals2, test_df, "Base FinBERT")

In [ ]:
import pandas as pd

df = pd.read_csv(path+"BTCUSDT.csv", sep='|')
df = df.rename(columns={'1502942400': 'timestamp', '4261.48': 'open', '4261.48.1': 'high', '4261.48.2': 'low', '4261.48.3': 'close', '1.775183': 'btc_volume_sell', '320.390851': 'usd_volume_buy', '0.075183': 'btc_volume_buy', '7564.906851': 'usd_volume_sell', '3': 'activity'})

df

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta

def backtest_sentiment_trading(
    model,
    tokenizer,
    df,
    text_data,
    initial_portfolio_usd=100_000,
    trade_fraction=0.5,
    hold_period=4,
    sentiment_treshold=0.2,
    minutes=525600
):
    model.eval()
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s')
    text_data['timestamp'] = pd.to_datetime(text_data['timestamp'], unit='s')

    df = df.sort_values('timestamp')
    text_data = text_data.sort_values('timestamp')

    start_time = df['timestamp'].max() - timedelta(minutes=minutes)
    df = df[df['timestamp'] >= start_time]
    text_data = text_data[text_data['timestamp'] >= start_time]

    portfolio_value = initial_portfolio_usd
    trades = []

    for _, row in text_data.iterrows():
        timestamp = row['timestamp']
        text = row['title']

        # Tokenize and run sentiment model
        inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
        with torch.no_grad():
            sentiment = model(
                input_ids=inputs["input_ids"].cuda(),
                attention_mask=inputs["attention_mask"].cuda()
            ).item()

        if abs(sentiment) < sentiment_treshold:
            continue  # No trade if sentiment is neutral

        #print(sentiment)

        entry_row = df[df['timestamp'] >= timestamp].head(1)
        exit_row = df[df['timestamp'] >= timestamp + timedelta(hours=hold_period)].head(1)

        if entry_row.empty or exit_row.empty:
            continue

        entry_price = entry_row['close'].values[0]
        exit_price = exit_row['close'].values[0]

        trade_usd = portfolio_value * trade_fraction
        btc_amount_to_trade = trade_usd / entry_price

        if sentiment > sentiment_treshold:
            pnl = btc_amount_to_trade * (exit_price - entry_price)  # Long
            action = "BUY"
        else:
            pnl = btc_amount_to_trade * (entry_price - exit_price)  # Short
            action = "SELL"

        portfolio_value += pnl

        trades.append({
            'time': timestamp,
            'action': action,
            'sentiment': sentiment,
            'entry_price': entry_price,
            'exit_price': exit_price,
            'btc_amount': btc_amount_to_trade,
            'pnl': pnl,
            'portfolio_value': portfolio_value
        })

    trades_df = pd.DataFrame(trades)

    # Save individual trades to CSV
    trades_df.to_csv(path + "trades_finbert_not_trained_all.csv", index=False)

    # Plot only if there are trades
    if not trades_df.empty:
        trades_df['cumulative_pnl'] = trades_df['portfolio_value'] - initial_portfolio_usd

        plt.figure(figsize=(12, 6))
        plt.plot(trades_df['time'], trades_df['cumulative_pnl'], marker='o', linestyle='-')
        plt.title("Cumulative Profit and Loss Over Time")
        plt.xlabel("Date")
        plt.ylabel("Cumulative PnL (USDT)")
        plt.grid(True)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

        print(f"Final Portfolio Value: {portfolio_value:.2f} USDT")
        print(f"Total PnL: {portfolio_value - initial_portfolio_usd:.2f} USDT from {len(trades_df)} trades")
    else:
        print("No trades executed during the backtest period.")

    return trades_df


In [ ]:
news_df = pd.read_csv(path+"output.csv")

trades = backtest_sentiment_trading(
    model=model,
    tokenizer=tokenizer,
    df=df,
    text_data=news_df,  # contains 'timestamp' and 'title'
    hold_period=4,
    sentiment_treshold=0.01,
    minutes=60*24*365  # 1 year
)

In [ ]:
news_df = pd.read_csv(path+"output.csv")

trades = backtest_sentiment_trading(
    model=model2,
    tokenizer=tokenizer,
    df=df,
    text_data=news_df,  # contains 'timestamp' and 'title'
    hold_period=2,
    sentiment_treshold=0.999,
    minutes=60*24*365  # 1 year
)